In [ ]:
# === SETUP ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import yfinance as yf
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plotting config
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries loaded")

## 1. Load Universe (310 Tickers)

In [ ]:
# Load ticker universe
tickers_df = pd.read_csv('../data/ticker_universe_300.csv')
tickers = tickers_df['ticker'].tolist()

print(f"📊 Universe: {len(tickers)} tickers")
print(f"Sectors: {tickers_df['sector'].value_counts().to_dict()}")

## 2. Fetch Historical Data (5 Years for Robust Testing)

In [ ]:
# Fetch 5 years of daily data
from datetime import datetime, timedelta

end_date = datetime.now()
start_date = end_date - timedelta(days=365*5)

print(f"Fetching data: {start_date.date()} to {end_date.date()}")
print(f"Estimated time: ~5 minutes for 310 tickers...\n")

# Use cache if available
cache_dir = Path('../data/daily_bars_cache')
cache_dir.mkdir(exist_ok=True)

all_data = {}
failed = []

for i, ticker in enumerate(tickers):
    cache_file = cache_dir / f"{ticker}_5y.csv"
    
    try:
        if cache_file.exists():
            df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
        else:
            df = yf.download(ticker, start=start_date, end=end_date, progress=False)
            if len(df) > 100:  # Minimum data requirement
                df.to_csv(cache_file)
        
        if len(df) > 100:
            all_data[ticker] = df
        
        if (i+1) % 50 == 0:
            print(f"  Loaded {i+1}/{len(tickers)}... ({len(all_data)} valid)")
            
    except Exception as e:
        failed.append(ticker)

print(f"\n✅ Loaded {len(all_data)} tickers")
print(f"❌ Failed: {len(failed)} tickers")
if failed:
    print(f"Failed list: {failed[:10]}...")

## 3. Calculate Momentum Features Across Multiple Horizons

In [ ]:
# Test momentum at different lookback/holding periods
lookback_periods = [5, 10, 20, 60, 120, 252]  # 1w, 2w, 1m, 3m, 6m, 1y
holding_periods = [5, 20, 60]  # 1w, 1m, 3m forward

momentum_results = []

for ticker, df in all_data.items():
    df = df.copy()
    
    # Calculate returns at multiple horizons
    for lookback in lookback_periods:
        df[f'ret_past_{lookback}d'] = df['Close'].pct_change(lookback)
    
    for holding in holding_periods:
        df[f'ret_future_{holding}d'] = df['Close'].pct_change(holding).shift(-holding)
    
    # Test all combinations
    for lookback in lookback_periods:
        for holding in holding_periods:
            past_col = f'ret_past_{lookback}d'
            future_col = f'ret_future_{holding}d'
            
            # Only use data where both are valid
            valid_data = df[[past_col, future_col]].dropna()
            
            if len(valid_data) > 100:
                # Linear regression: future_ret = α + β·past_ret
                X = sm.add_constant(valid_data[past_col])
                y = valid_data[future_col]
                model = sm.OLS(y, X).fit()
                
                # Autocorrelation
                autocorr = valid_data[past_col].corr(valid_data[future_col])
                
                momentum_results.append({
                    'ticker': ticker,
                    'lookback_days': lookback,
                    'holding_days': holding,
                    'beta': model.params[1],  # Momentum coefficient
                    'beta_tstat': model.tvalues[1],
                    'beta_pvalue': model.pvalues[1],
                    'r_squared': model.rsquared,
                    'autocorr': autocorr,
                    'n_obs': len(valid_data)
                })

results_df = pd.DataFrame(momentum_results)

print(f"\n✅ Tested {len(results_df)} combinations")
print(f"   {len(results_df['ticker'].unique())} tickers")
print(f"   {len(lookback_periods)} lookback periods")
print(f"   {len(holding_periods)} holding periods")
print(f"\nResults preview:")
results_df.head(10)

## 4. Test Universal Law: Is β > 0 for Majority?

In [ ]:
# Summary statistics across all tests
print("=" * 80)
print("MOMENTUM LAW VALIDATION")
print("=" * 80)

# Overall momentum prevalence
positive_beta = (results_df['beta'] > 0).mean()
significant_positive = ((results_df['beta'] > 0) & (results_df['beta_pvalue'] < 0.05)).mean()

print(f"\n📊 OVERALL RESULTS:")
print(f"   β > 0: {positive_beta:.1%} of tests")
print(f"   β > 0 AND significant: {significant_positive:.1%}")
print(f"   Mean β: {results_df['beta'].mean():.4f}")
print(f"   Median β: {results_df['beta'].median():.4f}")
print(f"   Mean R²: {results_df['r_squared'].mean():.4f}")

# Break down by lookback/holding period
print(f"\n📊 BY LOOKBACK PERIOD:")
for lookback in lookback_periods:
    subset = results_df[results_df['lookback_days'] == lookback]
    pct_pos = (subset['beta'] > 0).mean()
    mean_beta = subset['beta'].mean()
    mean_r2 = subset['r_squared'].mean()
    print(f"   {lookback:3d} days: β>0 = {pct_pos:.1%}, mean_β = {mean_beta:+.4f}, mean_R² = {mean_r2:.4f}")

print(f"\n📊 BY HOLDING PERIOD:")
for holding in holding_periods:
    subset = results_df[results_df['holding_days'] == holding]
    pct_pos = (subset['beta'] > 0).mean()
    mean_beta = subset['beta'].mean()
    mean_r2 = subset['r_squared'].mean()
    print(f"   {holding:3d} days: β>0 = {pct_pos:.1%}, mean_β = {mean_beta:+.4f}, mean_R² = {mean_r2:.4f}")

# Find optimal combination
print(f"\n🏆 BEST MOMENTUM STRATEGIES (by mean β):")
best = results_df.groupby(['lookback_days', 'holding_days']).agg({
    'beta': 'mean',
    'r_squared': 'mean',
    'ticker': 'count'
}).reset_index().sort_values('beta', ascending=False).head(5)
best.columns = ['lookback', 'holding', 'mean_beta', 'mean_r2', 'n_stocks']
print(best.to_string(index=False))

# Verdict
print(f"\n" + "=" * 80)
if positive_beta >= 0.7:
    print("✅ MOMENTUM LAW VALIDATED")
    print(f"   β > 0 for {positive_beta:.1%} of tests (threshold: 70%)")
    print("   Conclusion: Momentum is a UNIVERSAL force in markets")
elif positive_beta >= 0.5:
    print("⚠️  MOMENTUM LAW PARTIALLY VALIDATED")
    print(f"   β > 0 for {positive_beta:.1%} of tests (threshold: 70%)")
    print("   Conclusion: Momentum exists but is regime-dependent")
else:
    print("❌ MOMENTUM LAW REJECTED")
    print(f"   β > 0 for only {positive_beta:.1%} of tests")
    print("   Conclusion: Momentum is NOT a universal force")
print("=" * 80)

## 5. Visualize Momentum Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Distribution of beta coefficients
axes[0,0].hist(results_df['beta'], bins=50, alpha=0.7, edgecolor='black')
axes[0,0].axvline(0, color='red', linestyle='--', linewidth=2, label='β=0 (no momentum)')
axes[0,0].axvline(results_df['beta'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean β={results_df["beta"].mean():.3f}')
axes[0,0].set_xlabel('Beta Coefficient')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('Distribution of Momentum Coefficients (β)')
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# Plot 2: Beta vs R-squared (quality of momentum signal)
axes[0,1].scatter(results_df['beta'], results_df['r_squared'], alpha=0.3, s=10)
axes[0,1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0,1].set_xlabel('Beta (Momentum Strength)')
axes[0,1].set_ylabel('R² (Explanatory Power)')
axes[0,1].set_title('Momentum Strength vs Predictive Power')
axes[0,1].grid(alpha=0.3)

# Plot 3: Heatmap of beta by lookback/holding period
pivot = results_df.groupby(['lookback_days', 'holding_days'])['beta'].mean().unstack()
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=axes[1,0], 
            cbar_kws={'label': 'Mean β'})
axes[1,0].set_xlabel('Holding Period (days)')
axes[1,0].set_ylabel('Lookback Period (days)')
axes[1,0].set_title('Momentum Strength Heatmap')

# Plot 4: % of stocks with β > 0 by lookback period
pct_positive = results_df.groupby('lookback_days').apply(lambda x: (x['beta'] > 0).mean())
axes[1,1].bar(pct_positive.index, pct_positive.values, alpha=0.7, edgecolor='black')
axes[1,1].axhline(0.7, color='red', linestyle='--', linewidth=2, label='70% threshold')
axes[1,1].set_xlabel('Lookback Period (days)')
axes[1,1].set_ylabel('% Stocks with β > 0')
axes[1,1].set_title('Momentum Prevalence by Lookback Period')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3, axis='y')
axes[1,1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('../data/experiment_1_momentum_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved to data/experiment_1_momentum_visualization.png")

## 6. Test Regime Dependence (Does Momentum Work in All Markets?)

In [ ]:
# Fetch SPY to classify bull/bear markets
spy = yf.download('SPY', start=start_date, end=end_date, progress=False)
spy['sma_200'] = spy['Close'].rolling(200).mean()
spy['bull_market'] = spy['Close'] > spy['sma_200']

bull_days = spy['bull_market'].sum()
bear_days = (~spy['bull_market']).sum()

print(f"Market Regime Classification:")
print(f"  Bull days: {bull_days} ({bull_days/len(spy):.1%})")
print(f"  Bear days: {bear_days} ({bear_days/len(spy):.1%})")

# Now re-test momentum separately for bull vs bear periods
print(f"\nRe-testing momentum with regime filter...")
print(f"(This may take a few minutes...)\n")

regime_results = []

for ticker, df in list(all_data.items())[:100]:  # Test on subset for speed
    df = df.copy()
    
    # Merge SPY regime data
    df = df.join(spy[['bull_market']], how='left')
    df['bull_market'] = df['bull_market'].fillna(method='ffill')
    
    # Calculate 60-day momentum (optimal from above)
    df['ret_past_60d'] = df['Close'].pct_change(60)
    df['ret_future_20d'] = df['Close'].pct_change(20).shift(-20)
    
    for regime_name, regime_filter in [('Bull', True), ('Bear', False), ('All', None)]:
        if regime_filter is None:
            subset = df[['ret_past_60d', 'ret_future_20d']].dropna()
        else:
            subset = df[df['bull_market'] == regime_filter][['ret_past_60d', 'ret_future_20d']].dropna()
        
        if len(subset) > 50:
            X = sm.add_constant(subset['ret_past_60d'])
            y = subset['ret_future_20d']
            model = sm.OLS(y, X).fit()
            
            regime_results.append({
                'ticker': ticker,
                'regime': regime_name,
                'beta': model.params[1],
                'beta_pvalue': model.pvalues[1],
                'r_squared': model.rsquared,
                'n_obs': len(subset)
            })

regime_df = pd.DataFrame(regime_results)

# Compare beta across regimes
print("\n" + "="*80)
print("REGIME-DEPENDENT MOMENTUM")
print("="*80)
for regime in ['Bull', 'Bear', 'All']:
    subset = regime_df[regime_df['regime'] == regime]
    mean_beta = subset['beta'].mean()
    pct_positive = (subset['beta'] > 0).mean()
    mean_r2 = subset['r_squared'].mean()
    print(f"\n{regime} Market:")
    print(f"  Mean β: {mean_beta:+.4f}")
    print(f"  % β > 0: {pct_positive:.1%}")
    print(f"  Mean R²: {mean_r2:.4f}")
    print(f"  Sample: {len(subset)} stocks")

print("\n" + "="*80)

## 7. Build Momentum Portfolio (Top Decile Strategy)

In [ ]:
# Classic momentum strategy: Long top decile, short bottom decile
# Rebalance monthly

print("Building momentum portfolio backtest...")
print("Strategy: Long top 10% past 60-day returns, hold 20 days, rebalance monthly\n")

# Create universe returns matrix
returns_matrix = pd.DataFrame()
for ticker, df in all_data.items():
    returns_matrix[ticker] = df['Close'].pct_change()

# Calculate 60-day momentum for all stocks
momentum_60d = returns_matrix.rolling(60).apply(lambda x: (1 + x).prod() - 1)

# Backtest
portfolio_returns = []
rebalance_dates = momentum_60d.index[::20]  # Rebalance every 20 days

for date in rebalance_dates:
    if date not in momentum_60d.index:
        continue
    
    # Rank stocks by momentum
    mom_scores = momentum_60d.loc[date].dropna()
    
    if len(mom_scores) < 20:
        continue
    
    # Top 10% = winners, Bottom 10% = losers
    n_decile = max(1, len(mom_scores) // 10)
    winners = mom_scores.nlargest(n_decile).index.tolist()
    losers = mom_scores.nsmallest(n_decile).index.tolist()
    
    # Calculate forward returns (next 20 days)
    future_date_idx = returns_matrix.index.get_loc(date) + 20
    if future_date_idx >= len(returns_matrix):
        break
    
    future_returns = returns_matrix.iloc[returns_matrix.index.get_loc(date):future_date_idx+1]
    
    winner_ret = future_returns[winners].mean(axis=1).sum()
    loser_ret = future_returns[losers].mean(axis=1).sum()
    
    portfolio_returns.append({
        'date': date,
        'long_ret': winner_ret,
        'short_ret': -loser_ret,  # Short position
        'total_ret': winner_ret - loser_ret,
        'n_long': len(winners),
        'n_short': len(losers)
    })

portfolio_df = pd.DataFrame(portfolio_returns)

# Performance metrics
total_return = (1 + portfolio_df['total_ret']).prod() - 1
annual_return = (1 + total_return) ** (252 / len(portfolio_df)) - 1
sharpe = portfolio_df['total_ret'].mean() / portfolio_df['total_ret'].std() * np.sqrt(252/20)
win_rate = (portfolio_df['total_ret'] > 0).mean()

print("="*80)
print("MOMENTUM PORTFOLIO BACKTEST RESULTS")
print("="*80)
print(f"Total return: {total_return:.1%}")
print(f"Annualized return: {annual_return:.1%}")
print(f"Sharpe ratio: {sharpe:.2f}")
print(f"Win rate: {win_rate:.1%}")
print(f"Number of rebalances: {len(portfolio_df)}")
print(f"Avg positions: {portfolio_df['n_long'].mean():.0f} long, {portfolio_df['n_short'].mean():.0f} short")
print("="*80)

# Plot equity curve
portfolio_df['cumulative_ret'] = (1 + portfolio_df['total_ret']).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(portfolio_df['date'], portfolio_df['cumulative_ret'], linewidth=2, label='Momentum Strategy')
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Date')
plt.ylabel('Cumulative Return (1.0 = breakeven)')
plt.title('Momentum Portfolio Equity Curve (Long Top 10%, Short Bottom 10%)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/experiment_1_momentum_portfolio.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Portfolio backtest complete")

## 8. CONCLUSIONS & NEXT STEPS

In [ ]:
print("="*80)
print("EXPERIMENT 1 CONCLUSIONS")
print("="*80)

print("\n1. IS MOMENTUM A UNIVERSAL LAW?")
if positive_beta >= 0.7:
    print("   ✅ YES - β > 0 for majority of stocks")
else:
    print("   ⚠️  CONDITIONAL - Works in specific regimes only")

print("\n2. OPTIMAL PARAMETERS:")
print(f"   Lookback: {best.iloc[0]['lookback']:.0f} days")
print(f"   Holding: {best.iloc[0]['holding']:.0f} days")
print(f"   Mean β: {best.iloc[0]['mean_beta']:.4f}")

print("\n3. REGIME DEPENDENCE:")
bull_beta = regime_df[regime_df['regime'] == 'Bull']['beta'].mean()
bear_beta = regime_df[regime_df['regime'] == 'Bear']['beta'].mean()
print(f"   Bull market β: {bull_beta:+.4f}")
print(f"   Bear market β: {bear_beta:+.4f}")
if bull_beta > bear_beta:
    print("   Conclusion: Momentum STRONGER in bull markets")
else:
    print("   Conclusion: Momentum works in all regimes")

print("\n4. TRADEABLE EDGE:")
if sharpe > 1.0:
    print(f"   ✅ YES - Sharpe {sharpe:.2f} (institutional grade)")
    print(f"   Strategy: Long/short momentum portfolio")
elif sharpe > 0.5:
    print(f"   ⚠️  MARGINAL - Sharpe {sharpe:.2f} (needs refinement)")
else:
    print(f"   ❌ NO - Sharpe {sharpe:.2f} (not tradeable)")

print("\n5. NEXT EXPERIMENTS:")
print("   - Experiment 2: Volatility clustering (GARCH)")
print("   - Experiment 3: Information propagation (lead-lag)")
print("   - Experiment 4: Network effects (correlation structure)")
print("   - Experiment 5: Regime switching (adaptive momentum)")

print("\n" + "="*80)
print("END OF EXPERIMENT 1")
print("="*80)